# Exploration scratchpad

A workbench for inspecting query results and iterating on Altair encodings.
Not a deliverable: when the charts settle, `quarto convert explore.ipynb`
turns this into a `.qmd` to write the narrative around.

Start it with:

```
uv run jupyter lab
```

**Restart-and-run-all before trusting anything here.** Out-of-order execution
is the classic way a notebook ends up depending on a cell you deleted.

In [ ]:
import sys
from pathlib import Path

# scripts/ is not an installed package, so put the repo root on sys.path to let
# `from scripts.queries import ...` resolve. Path.cwd() is notebooks/ when Jupyter
# is launched from the repo root, so the root is its parent.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import altair as alt
import pandas as pd

from scripts.queries import ALTAIR_MAX_ROWS, ANALYSIS_QUERIES, run_query, run_script

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. What shape is every query result?

Row count decides chart type. Anything over `ALTAIR_MAX_ROWS` cannot be plotted
row-per-record and needs aggregating in SQL first.

In [ ]:
results = {name: run_query(name) for name in ANALYSIS_QUERIES}

pd.DataFrame(
    [
        {
            "query": name,
            "rows": len(df),
            "cols": len(df.columns),
            "over_altair_limit": len(df) > ALTAIR_MAX_ROWS,
        }
        for name, df in results.items()
    ]
)

## 2. Inspect one result

Check dtypes here. SQLite is loose about types, so a `CASE` branch that returns
a string somewhere will show up as `object` where you expected `float64`.

In [ ]:
df = results["product_summary"]
print(df.dtypes)
df.head()

## 3. The integrity checks

`data_profiling.sql` holds three independent statements, so it needs
`run_script()`. `run_query()` would silently return only the first.

In [ ]:
for i, check in enumerate(run_script("data_profiling"), start=1):
    print(f"--- statement {i} ---")
    print(check.to_string(index=False))
    print()

## 4. First Altair chart

A throwaway example to confirm charts render inline. `product_summary` is already
`LIMIT 10`, so it is safely under the row cap.

The pattern is: map columns to encodings, and let Altair infer the rest. The
suffixes declare types - `:Q` quantitative, `:N` nominal, `:T` temporal.

In [ ]:
alt.Chart(results["product_summary"]).mark_bar().encode(
    x=alt.X("review_count:Q", title="Reviews"),
    y=alt.Y("product_name:N", sort="-x", title=None),
    tooltip=["product_name:N", "review_count:Q", "avg_stars:Q"],
).properties(height=260, title="Most-reviewed products")

## 5. Scratch

Everything below here is disposable.